# Introdução à Localização de Objetos

A Localização de Objetos (Object Localization) é uma tarefa fundamental em Visão Computacional que estende a classificação de imagens. Enquanto a classificação determina *o que* está na imagem, a localização determina *onde* o objeto está, geralmente definindo uma **caixa delimitadora (bounding box)** ao seu redor.

Neste notebook, abordaremos a localização de **um único objeto** (um dígito do MNIST) por imagem. Aprenderemos sobre os diferentes formatos de bounding boxes, criaremos um modelo com duas cabeças (classificação e regressão) e definiremos uma função de perda composta para treinar nossa rede.

In [ ]:
import os
import glob
import random
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Bounding Boxes

As **bounding boxes** (caixas delimitadoras) representam a localização de um objeto em uma imagem. Existem dois formatos principais para representá-las:

1. **Formato Pixels $[x_0, y_0, x_1, y_1]$**: Representa as coordenadas $(x, y)$ do canto superior esquerdo e do canto inferior direito da caixa em pixels. É útil para plotagem e corte (crop) da imagem.
2. **Formato YOLO/Normalizado $[c_x, c_y, w, h]$**: Representa as coordenadas do centro da caixa, e a largura e altura da caixa. Todos os valores são **normalizados** (divididos pela largura ou altura da imagem) para ficarem no intervalo $[0, 1]$. Este é o formato padrão mais eficaz para treinamento de redes neurais, pois estabiliza os gradientes.

As conversões matemáticas entre os formatos, considerando uma imagem de dimensões $W \times H$, são:

**De Pixels para YOLO:**
$$ c_x = \frac{x_0 + x_1}{2W}, \quad c_y = \frac{y_0 + y_1}{2H} $$
$$ w = \frac{x_1 - x_0}{W}, \quad h = \frac{y_1 - y_0}{H} $$

**De YOLO para Pixels:**
$$ x_0 = (c_x - \frac{w}{2}) \times W, \quad y_0 = (c_y - \frac{h}{2}) \times H $$
$$ x_1 = (c_x + \frac{w}{2}) \times W, \quad y_1 = (c_y + \frac{h}{2}) \times H $$

In [ ]:
def yolo_to_pixels(box_yolo, img_size):
    """Converte de [cx, cy, w, h] (normalizado) para [x0, y0, w_pixel, h_pixel]"""
    cx, cy, w, h = box_yolo
    img_w, img_h = img_size
    
    w_px = w * img_w
    h_px = h * img_h
    x0 = (cx - w / 2) * img_w
    y0 = (cy - h / 2) * img_h
    return [x0, y0, w_px, h_px]

In [ ]:
import gdown
import zipfile

gdown.download(f"https://drive.google.com/uc?id=1HLolOC_nBb-lQ7EEGj7wirK-7OKX9-eq", "mnist_localization.zip", quiet=False)

with zipfile.ZipFile("mnist_localization.zip") as z:
    z.extractall("data")

In [ ]:
# Exemplo: Carregando e visualizando uma imagem do dataset gerado
img_path = 'data/mnist_localization/train/images/00000.png'
txt_path = 'data/mnist_localization/train/labels/00000.txt'

img = Image.open(img_path)
with open(txt_path, 'r') as f:
    line = f.readline().strip().split()
    label = int(line[0])
    box_yolo = [float(x) for x in line[1:]]
    
fig, ax = plt.subplots(1)
ax.imshow(img, cmap='gray')

# Desenhar Bounding Box
x0, y0, w_px, h_px = yolo_to_pixels(box_yolo, img.size)
rect = patches.Rectangle((x0, y0), w_px, h_px, linewidth=2, edgecolor='r', facecolor='none')
ax.add_patch(rect)
ax.set_title(f'Dígito: {label} | Box YOLO: {[round(x, 2) for x in box_yolo]}')
plt.show()

### Preparação dos Dados

Nesta seção, criaremos um `Dataset` customizado no PyTorch para carregar nossas imagens sintéticas de MNIST e as respectivas labels em formato de texto. Como o objetivo é de localização simples, assumiremos que existe apenas **um** objeto por imagem ($N=1$).

In [ ]:
class MNISTLocalizationDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.images_dir = os.path.join(data_dir, 'images')
        self.labels_dir = os.path.join(data_dir, 'labels')
        self.transform = transform
        self.img_files = sorted(os.listdir(self.images_dir))
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        label_path = os.path.join(self.labels_dir, img_name.replace('.png', '.txt'))
        
        # Carrega imagem
        image = Image.open(img_path) #.convert('RGB')
        
        # Carrega label (YOLO format: class_id cx cy w h)
        with open(label_path, 'r') as f:
            line = f.readline().strip().split()
            label = int(line[0])
            bbox = [float(x) for x in line[1:]]
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long), torch.tensor(bbox, dtype=torch.float32)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = MNISTLocalizationDataset('data/mnist_localization/train', transform=transform)
val_dataset = MNISTLocalizationDataset('data/mnist_localization/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'Tamanho do treino: {len(train_dataset)}')
print(f'Tamanho da validação: {len(val_dataset)}')

In [ ]:
# Exibindo um batch
imgs, labels, bboxes = next(iter(train_loader))
print(f'Shape Imagens: {imgs.shape} | Shape Labels: {labels.shape} | Shape BBoxes: {bboxes.shape}')

### Arquitetura do Modelo

Nosso modelo será uma **Rede Neural Convolucional (CNN)** simples com duas saídas (também chamadas de *heads* ou cabeças):
1. **Cabeça de Classificação (`cls_head`)**: Prevê a probabilidade do dígito ser de 0 a 9.
2. **Cabeça de Regressão (`reg_head`)**: Prevê as 4 coordenadas normalizadas da *bounding box* $[c_x, c_y, w, h]$. Usamos a função de ativação `Sigmoid` no final desta cabeça para garantir que os valores previstos fiquem estritamente entre 0 e 1, que é o formato YOLO.

A parte inicial da rede (o *backbone*) extrairá características da imagem, e o tensor resultante será achatado (flatten) e alimentado para essas duas cabeças.

In [ ]:
class SimpleLocalizationNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Backbone CNN simples
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 112 -> 56
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 56 -> 28
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 28 -> 14
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 14 -> 7
        )
        
        # Dimension após backbone: batch_size x 128 x 7 x 7
        flatten_size = 128 * 7 * 7
        
        # Classificador (Classe do Dígito)
        self.cls_head = nn.Sequential(
            nn.Linear(flatten_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
        # Regressor (Bounding Box)
        self.reg_head = nn.Sequential(
            nn.Linear(flatten_size, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Sigmoid() # Saída entre 0 e 1 (YOLO)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten
        
        logits = self.cls_head(x)
        bbox = self.reg_head(x)
        
        return logits, bbox

### Função de Perda (Loss)

Para treinar o modelo multi-tarefa, precisamos de uma função de perda combinada:

$$ L_{total} = L_{cls} + \lambda L_{reg} $$

1. **Perda de Classificação ($L_{cls}$)**: Usaremos a função `CrossEntropyLoss` padrão para prever a classe de 0 a 9.
2. **Perda de Regressão ($L_{reg}$)**: Usaremos a função de Erro Médio Absoluto (L1 Loss, `nn.L1Loss()`) ou Erro Quadrático Médio (MSE). O `L1Loss` costuma ser mais robusto para localização pois penaliza menos os *outliers*.

O hiperparâmetro $\lambda$ controla o peso da regressão em relação à classificação. Ajustar este parâmetro é essencial quando as perdas têm magnitudes muito diferentes.

In [ ]:
class LocLoss(nn.Module):
    def __init__(self, lambda_reg=5.0):
        super().__init__()
        self.lambda_reg = lambda_reg
        self.cls_loss_fn = nn.CrossEntropyLoss()
        self.reg_loss_fn = nn.L1Loss()
        
    def forward(self, pred_logits, pred_bboxes, true_labels, true_bboxes):
        l_cls = self.cls_loss_fn(pred_logits, true_labels)
        l_reg = self.reg_loss_fn(pred_bboxes, true_bboxes)
        
        total_loss = l_cls + self.lambda_reg * l_reg
        return total_loss, l_cls, l_reg

### Otimizadores e Loop de Treinamento

Implementamos um loop padrão do PyTorch que iterará sobre nossos minibatches, efetuará o backpropagation e acumulará o custo ao longo de algumas épocas.

In [ ]:
model = SimpleLocalizationNet().to(device)
criterion = LocLoss(lambda_reg=5.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
num_epochs = 10
history = {
    'loss': [],
    'cls_loss': [],
    'reg_loss': []
}

for epoch in range(num_epochs):
    model.train()
    
    running_loss = 0.0
    running_cls_loss = 0.0
    running_reg_loss = 0.0
    
    loop = tqdm(train_loader, leave=False)
    for imgs, labels, bboxes in loop:
        imgs = imgs.to(device)
        labels = labels.to(device)
        bboxes = bboxes.to(device)
        
        optimizer.zero_grad()
        pred_logits, pred_bboxes = model(imgs)
        
        loss, l_cls, l_reg = criterion(pred_logits, pred_bboxes, labels, bboxes)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        running_cls_loss += l_cls.item()
        running_reg_loss += l_reg.item()
        
        loop.set_description(f'Epoch [{epoch+1}/{num_epochs}]')
        loop.set_postfix(
            loss=loss.item(),
            cls=l_cls.item(),
            reg=l_reg.item()
        )
        
    epoch_loss = running_loss / len(train_loader)
    epoch_cls_loss = running_cls_loss / len(train_loader)
    epoch_reg_loss = running_reg_loss / len(train_loader)
    
    history['loss'].append(epoch_loss)
    history['cls_loss'].append(epoch_cls_loss)
    history['reg_loss'].append(epoch_reg_loss)
    
    print(
        f'Epoch [{epoch+1}/{num_epochs}] - '
        f'Mean Loss: {epoch_loss:.4f} | '
        f'Cls Loss: {epoch_cls_loss:.4f} | '
        f'Reg Loss: {epoch_reg_loss:.4f}'
    )

In [ ]:
plt.plot(history['loss'], label='Total Loss')
plt.plot(history['cls_loss'], label='Classification Loss')
plt.plot(history['reg_loss'], label='Regression Loss')

plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Geração, Avaliação e Visualização

Por fim, utilizaremos nosso modelo treinado para fazer inferência sobre imagens do conjunto de validação e desenharemos as caixas preditas (em vermelho) comparadas com as verdadeiras (em verde).

In [ ]:
model.eval()

# Pega um batch de validação
imgs, labels, bboxes = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    pred_logits, pred_bboxes = model(imgs)

# Predições finais
pred_labels = torch.argmax(pred_logits, dim=1).cpu().numpy()
pred_bboxes = pred_bboxes.cpu().numpy()
true_labels = labels.numpy()
true_bboxes = bboxes.numpy()
imgs = imgs.cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    # Desnormalizar imagem
    img_plot = imgs[i].transpose(1, 2, 0) * 0.5 + 0.5
    ax.imshow(img_plot, cmap='gray')
    
    # Desenhar Bounding Box verdadeira (Verde)
    x0_t, y0_t, w_t, h_t = yolo_to_pixels(true_bboxes[i], (112, 112))
    rect_t = patches.Rectangle((x0_t, y0_t), w_t, h_t, linewidth=2, edgecolor='g', facecolor='none')
    ax.add_patch(rect_t)
    
    # Desenhar Bounding Box predita (Vermelha)
    x0_p, y0_p, w_p, h_p = yolo_to_pixels(pred_bboxes[i], (112, 112))
    rect_p = patches.Rectangle((x0_p, y0_p), w_p, h_p, linewidth=2, edgecolor='r', facecolor='none', linestyle='--')
    ax.add_patch(rect_p)
    
    ax.set_title(f'GT: {true_labels[i]} | Pred: {pred_labels[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Exercícios

### Exercício 1: Arquitetura do Modelo

Modifique a classe `SimpleLocalizationNet` para substituir o backbone convolucional básico por modelo pré-treinado e avalie se o aumento da capacidade de extração de características do modelo resulta em bounding boxes mais precisas.